# Time-aware string-edit parameter sweep

This notebook is only a launcher and results viewer. The parallel, resumable implementation lives in `benchmarks/modules/note/StringEditParamSweep.py`; the command-line entry point is `benchmarks/sweeps/scripts/stringedit_param_sweep.py`. Worker processes operate at track granularity: each worker loads one cached smoothed-pYIN track, detects notes once, then evaluates the complete parameter list. Raw rows and summaries are checkpointed under `benchmarks/sweeps/results/mistake`, so restarting this notebook does not discard completed stages.

The swept pairing cost is

$$C_{pitch}=|p_u-p_s|$$

$$C_{time}=\alpha_1|t_u-t_s|+\alpha_2|d_u-d_s|,\qquad \alpha_1+\alpha_2=1$$

$$C_{pair}=\gamma_1C_{pitch}+\gamma_2C_{time}.$$

The same workers also run a balanced symbolic mixture of substitution, deletion, insertion, short, and long mistakes. Selection is constrained not to regress from production on clean-Coco false mistakes/indels or on mixed-error accuracy, correct-note preservation, and source-pair recall.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'app.py').is_file() and (path / 'benchmarks').is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmarks.modules.note.StringEditParamSweep import (
    PARAM_COLUMNS, accepted_vs_production, load_results, production_params,
)

OUTPUT_DIR = REPO_ROOT / 'benchmarks' / 'sweeps' / 'results' / 'mistake'
WORKERS = max(1, (os.cpu_count() or 4) - 1)
FORCE_RECOMPUTE = False
print('repo:', REPO_ROOT)
print('workers:', WORKERS)
print('results:', OUTPUT_DIR)

## Run or resume

The command runs outside the notebook kernel, so macOS process spawning is reliable and a kernel restart does not affect it. Leave `FORCE_RECOMPUTE=False` to reuse any stage whose signature still matches the selected tracks, parameters, and runner version.

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / 'benchmarks' / 'sweeps' / 'scripts' / 'stringedit_param_sweep.py'),
    '--workers', str(WORKERS),
    '--output-dir', str(OUTPUT_DIR),
    '--seed', '0',
]
if FORCE_RECOMPUTE:
    command.append('--force')
print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

## Load results

These cells perform no DSP or alignment work. They read the checkpointed CSV summaries and can be rerun independently after reopening the notebook.

In [ ]:
results = load_results(OUTPUT_DIR)
sample = results['sample']
track_meta = results['track_meta']
tune = results['tune']
holdout = results['holdout']
recommended = results['recommendation']

print(f"sample: {len(sample)} stems across {sample.groupby(['ensemble', 'instrument']).ngroups} strata")
display(sample.groupby(['ensemble', 'instrument', 'role']).size().unstack(fill_value=0))
truth_columns = [column for column in track_meta if column.startswith('truth_')]
print('mixed symbolic truth counts:')
display(track_meta[truth_columns].fillna(0).sum().astype(int).to_frame('events').T)
display(track_meta.groupby(['ensemble', 'instrument', 'role']).agg(
    tracks=('track_id', 'size'),
    reference_notes=('reference_notes', 'sum'),
    estimated_notes=('estimated_notes', 'sum'),
    pitch_aware_note_f1=('pitch_aware_note_f1', 'mean'),
))

In [ ]:
RESULT_COLUMNS = PARAM_COLUMNS + [
    'pair_f1', 'false_mistakes_per_ref', 'indels_per_ref',
    'mixed_source_pair_recall', 'mixed_error_f1', 'mixed_pitch_f1',
    'mixed_duration_f1', 'mixed_correct_f1', 'mixed_indels_per_event',
]

def parameter_mask(frame, params):
    return np.logical_and.reduce([frame[name].eq(value) for name, value in params.items()])

print('Production on tuning sample:')
display(tune.loc[parameter_mask(tune, production_params()), RESULT_COLUMNS])
print('Best production-safe tuning candidates:')
display(accepted_vs_production(tune)[RESULT_COLUMNS].head(20))
print('Unconstrained pair-F1 leaders (diagnostic):')
display(tune.sort_values('pair_f1', ascending=False)[RESULT_COLUMNS].head(10))

In [ ]:
accepted_holdout = accepted_vs_production(holdout)
print('All held-out candidates:')
display(holdout.sort_values(
    ['pair_f1', 'mixed_error_f1'], ascending=[False, False]
)[RESULT_COLUMNS])
print('Production-safe held-out candidates:')
display(accepted_holdout[RESULT_COLUMNS])
print('Recommended configuration:')
print('\n'.join(f'{name}={recommended[name]:g}' for name in PARAM_COLUMNS))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
points = ax.scatter(
    tune.pair_f1, tune.mixed_error_f1,
    c=tune.false_mistakes_per_ref, cmap='viridis_r', alpha=0.65, s=28,
)
production_row = tune.loc[parameter_mask(tune, production_params())].iloc[0]
recommended_row = tune.loc[parameter_mask(tune, recommended)].iloc[0]
ax.scatter(production_row.pair_f1, production_row.mixed_error_f1,
           marker='X', s=180, color='red', label='production')
ax.scatter(recommended_row.pair_f1, recommended_row.mixed_error_f1,
           marker='*', s=220, color='gold', edgecolor='black', label='recommended')
ax.set(xlabel='Coco timing-pair F1', ylabel='Exact mixed-mistake F1',
       title='Alignment quality versus mixed-error accuracy')
ax.legend()
fig.colorbar(points, ax=ax, label='Clean-Coco false mistakes per reference note')
fig.tight_layout()

## Interpretation

- `pair_f1` compares dynamic-programming diagonal pairs with a maximum timing-valid, pitch-agnostic note matching.
- `false_mistakes_per_ref` and `indels_per_ref` are guardrails measured on clean rendered Coco stems.
- `mixed_error_f1` scores substitution, deletion, insertion, short, and long as exact labels. A substitution represented as deletion+insertion receives no substitution credit.
- `mixed_correct_f1` checks that unaffected notes remain correct; `mixed_source_pair_recall` checks that surviving performed notes remain paired to their originating score notes.
- Metrics are averaged within each `(ensemble, instrument)` stratum before strata are averaged, so large strata cannot dominate.
- The recommendation is selected only from held-out configurations that do not regress from production on the clean-Coco or mixed-symbolic guardrails. Coco's available caches are from its test manifest, so final paper reporting still requires a separate untouched corpus or newly cached validation split.